<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/05_data_storytelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Storytelling

# Demo: A Practical Data Storytelling Workflow

This notebook is a practical companion to the Data Storytelling lesson. We will follow the structured workflow outlined in the lesson to analyze the King County House Sales dataset and build a data story for a specific audience.

**Our Tools:**
* **Polars:** For high-performance data manipulation.
* **Altair:** For declarative and expressive statistical visualizations.

Let's begin our journey!

## Setup: Imports and Data Creation

First, let's import our libraries and create the `kc_house_data.csv` file. For this demo, we'll create the file directly from a string so the notebook is self-contained.


In [ ]:
import polars as pl
import altair as alt
import pandas as pd # Altair works well with pandas, so it's good to have it available
import functools

In [ ]:
file_path = "/content/drive/MyDrive/dataprogpy/data/kc_house_data.csv"

## Phase 1: Understanding the Landscape & Defining Goals

As the lesson states, every data story begins with understanding our data and defining clear goals.

### Step 1 & 2: Define an Audience Persona & Analytical Goals

We'll adopt the persona from the lesson: **"Sarah the Savvy Homebuyer"**.

* **Role:** A working professional looking to buy her first home.
* **Goal:** To understand the King County housing market to find a 2-3 bedroom house that balances affordability, good condition, and reasonable commute.
* **Key Questions:**
    1.  What is the general price distribution for houses? What's a "typical" price?
    2.  How much do key features like living area (`sqft_living`), `bedrooms`, or `bathrooms` impact the `price`?
    3.  Are there significant price differences between `zipcode`s? Which areas are more affordable?
    4.  Is there a premium for better `grade` (construction quality)?

### Step 3: Initial Data Visualization Task List (V1)

Based on Sarah's questions, we'll create a V1 of our task list. This is our initial plan for the visualizations we want to create.

| Research Question                                       | Data Variables                     | Mark Type    | Key Encodings (X, Y, Color, etc.)                               | Anticipated Chart Type |
| :------------------------------------------------------ | :--------------------------------- | :----------- | :-------------------------------------------------------------- | :--------------------- |
| 1. What is the general `price` distribution?            | `price`                            | `bar`        | `x: price` (binned), `y: count()`                               | Histogram              |
| 2. How does `sqft_living` relate to `price`?            | `price`, `sqft_living`             | `point`      | `x: sqft_living`, `y: price`                                    | Scatter Plot           |
| 3. What's the median `price` by `zipcode`?              | `price`, `zipcode`                 | `bar`        | `x: zipcode`, `y: median(price)`                                | Bar Chart              |
| 4. How does `grade` affect `price`?                     | `price`, `grade`                   | `boxplot`    | `x: grade`, `y: price`                                          | Box Plot               |

## Phase 2: Data Ingestion and Initial Exploration

Now we load the data and perform our first "reality check".



### Step 4: Ingesting Data with Polars & Initial Inspection

In [ ]:
# Load the CSV file into a Polars DataFrame
try:
    housing_df = pl.read_csv(file_path)
    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset: {e}")
    housing_df = pl.DataFrame()

In [ ]:
housing_df.head()

In [ ]:
housing_df.schema

**Initial Observations:**
- `date` is a string (`Utf8`), not a datetime object. This will need to be fixed if we want to do time-based analysis.
- `zipcode` is an integer, which is okay, but we must remember to treat it as a category, not a number to be averaged.

In [ ]:
housing_df.describe()

**Performing a Quick Data "Smoke Test"**

Let's check a few critical columns for anything obviously wrong.




In [ ]:
# Check for nulls in key columns
print(f"Nulls in 'price': {housing_df['price'].is_null().sum()}")
print(f"Nulls in 'sqft_living': {housing_df['sqft_living'].is_null().sum()}")

# Check for non-positive prices
non_positive_prices = housing_df.filter(pl.col('price') <= 0).height
print(f"Records with non-positive prices: {non_positive_prices}")


In [ ]:
# Check unique values for a categorical-like column
unique_bedrooms = housing_df['bedrooms'].unique().sort()
print(f"Unique values in 'bedrooms': {unique_bedrooms.to_list()}")

unique_bathrooms = housing_df['bathrooms'].unique().sort()
print(f"Unique values in 'bathrooms': {unique_bathrooms.to_list()}")

unique_floors = housing_df['floors'].unique().sort()
print(f"Unique values in 'floors': {unique_floors.to_list()}")

unique_grade = housing_df['grade'].unique().sort()
print(f"Unique values in 'grade': {unique_grade.to_list()}")

unique_waterfront = housing_df['waterfront'].unique().sort()
print(f"Unique values in 'waterfront': {unique_waterfront.to_list()}")

The data passes our initial smoke test. No nulls in key columns and no major logical errors found in this quick check. We can proceed to our first visualization.

### Step 5: First Pass Visualization & The "Reality Check"

Let's try to create the scatter plot from our task list (V1): `price` vs. `sqft_living`. This is our "reality check".

#### Price vs. Living Area

In [ ]:
alt.data_transformers.disable_max_rows()

In [ ]:
scatter_plot_v1 = alt.Chart(housing_df).mark_point().encode(
    x=alt.X('sqft_living', title='Living Area (sq ft)'),
    y=alt.Y('price', title='Sale Price (USD)'),
    tooltip=['price', 'sqft_living', 'bedrooms', 'bathrooms']
).properties(
    title='V1 Reality Check: Price vs. Living Area',
    width=600,
    height=400
).interactive()

display(scatter_plot_v1)

#### Distributions of Key Properties

In [ ]:
sqft_living_histogram = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('sqft_living', bin=alt.Bin(maxbins=30), title='Living Area (sq ft)'),
    y=alt.Y('count()', title='Number of Houses'),
    tooltip=[
        alt.Tooltip('count()', title='Number of Houses'),
        alt.Tooltip('sqft_living', title='Living Area (sq ft)')
    ]

).properties(
    title='V1 Reality Check: Houses by Living Area',
)

price_histogram = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('price', bin=alt.Bin(maxbins=30), title='Sale Price (USD)'),
    y=alt.Y('count()', title='Number of Houses'),
    tooltip=[
        alt.Tooltip('count()', title='Number of Houses'),
        alt.Tooltip('price', title='Sale Price (USD)')
    ]
).properties(
    title='V1 Reality Check: Houses by Sale Price',
)
price_histogram | sqft_living_histogram


In [ ]:
bedrooms_v1 = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('bedrooms', title='Number of Bedrooms'),
    y=alt.Y('count()', title='Number of Houses'),
).properties(
    title='V1 Reality Check: Houses by Number of Bedrooms',
)

bathrooms_v1 = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('bathrooms', title='Number of Bathrooms'),
    y=alt.Y('count()', title='Number of Houses'),
).properties(
    title='V1 Reality Check: Houses by Number of Bathrooms',
)

floors_v1 = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('floors', title='Number of Floors'),
    y=alt.Y('count()', title='Number of Houses'),
).properties(
    title='V1 Reality Check: Houses by Number of Floors',
)

waterfront_v1 = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('waterfront', title='Waterfront'),
    y=alt.Y('count()', title='Number of Houses'),
).properties(
    title='V1 Reality Check: Houses with Waterfront',
)

(bedrooms_v1 | bathrooms_v1) & (floors_v1 | waterfront_v1)

In [ ]:
grade_histogram = alt.Chart(housing_df).mark_bar().encode(
    x=alt.X('grade', title='Construction Grade'),
    y=alt.Y('count()', title='Number of Houses'),
).properties(
    title='V1 Reality Check: Houses by Construction Grade',
)

grade_pie = alt.Chart(housing_df).mark_arc(innerRadius=50).encode(
    theta="count()",
    color="grade:N",
    tooltip=[
        alt.Tooltip('count()', title='Number of Houses'),
        alt.Tooltip('grade', title='Construction Grade')
    ]
).properties(
    title='V1 Reality Check: Houses by Construction Grade (Pie Chart)',
)

grade_histogram | grade_pie

### The "Uh-Oh" Moment 😲

Our V1 scatter plot reveals several problems:

1.  **Skewed Distribution:** The several key variables are heavily right-skewed. A few very expensive houses "squish" the majority of the data points into the bottom of the chart, making it hard to see the relationship for typical homes.
2.  **Overplotting:** Even with our small dataset, points are plotted on top of each other, obscuring the true density.
3.  **Lack of Context:** Is the relationship the same for all types of houses? We can't tell from this chart.

These observations directly inform our next steps. We need to clean and transform the data to make a more effective visualization.

---
## Phase 3: Iterative Cleaning and Visualization

Now we respond to our "Uh-Oh" moments by creating a cleaning task list and executing on it.

### Step 6: Responsive Data Cleaning Task List

| "Uh-Oh" / Observation                        | Data Variable(s)      | Proposed Cleaning Task                                      | Rationale (for Sarah)                                             |
| :------------------------------------------- | :-------------------- | :---------------------------------------------------------- | :---------------------------------------------------------------- |
| Price vs. Sqft plot is skewed and hard to read.  | `price`               | Apply a log transform to `price` to create `price_log`.     | Better shows the relationship for mid-range homes, which Sarah cares about. |
| We need a normalized value metric.           | `price`, `sqft_living`| Create `price_per_sqft`.                                    | Helps Sarah compare the "value for money" between different houses. |
| Date is a string, preventing trend analysis. | `date`                | Parse `date` into a proper datetime object.                 | Allows future analysis of market trends over time.               |
.
.
.

### Step 7: Executing Cleaning and Updating Visualizations

Let's execute our cleaning tasks with Polars.

In [ ]:
Beds = pl.Enum(["LessThan_1", "1+", "2+", "3+", "4+", "5+" ])
Baths = pl.Enum(["LessThan_1", "1+", "2+", "3+"])
Floors = pl.Enum("1+ 2+ 3+".split())
Grades = pl.Enum("1 2 3 4 5 6 7 8 9 10 11 12 13".split())

def num_to_cat_mapper(val: int|float, max: int)-> str:
    if val > max:
      return f"{max}+"
    elif val < 1:
        return "LessThan_1"
    elif val <= 1:
        return "1+"
    elif val <= 2:
        return "2+"
    elif val <= 3:
        return "3+"
    elif val <= 4:
        return "4+"
    else:
        return "5+"

bed_mapper = functools.partial(num_to_cat_mapper, max=5)
bath_mapper = functools.partial(num_to_cat_mapper, max=3)
floor_mapper = functools.partial(num_to_cat_mapper, max=3)

In [ ]:
# Execute Cleaning Tasks
housing_cleaned_df = housing_df.with_columns(
  pl.col('price', 'sqft_living').log().name.suffix("_log"),
  pl.col('grade').cast(pl.String),
  pl.col('grade').cast(pl.String).cast(Grades).alias('grade_cat'),
  pl.col('zipcode').cast(pl.String).cast(pl.Categorical),
  pl.when(pl.col('sqft_living') > 0)
    .then(pl.col('price') / pl.col('sqft_living'))
    .otherwise(None)
    .alias('price_per_sqft'),
  pl.col('date').str.strptime(pl.Date, format="%Y%m%dT%H%M%S").alias('sale_date'),
  pl.col('bedrooms').map_elements(
      bed_mapper,
      return_dtype=pl.String).cast(Beds).alias("bed_cat"),
  pl.col('bathrooms').map_elements(
      bath_mapper,
      return_dtype=pl.String).cast(Baths).alias("bath_cat"),
  pl.col('floors').map_elements(
      floor_mapper,
      return_dtype=pl.String).cast(Floors).alias("floor_cat")
)

In [ ]:
print("Cleaning and feature engineering complete. New columns added:")
display(housing_cleaned_df.select(['price', 'price_log', 'sqft_living_log', 'price_per_sqft', 'date', 'sale_date', "bed_cat", "bath_cat", "floor_cat"]).head())

### "Vertical Visual Development": Refining Our Charts

Now, let's rebuild our visualizations using the cleaned data and add layers of information.

**Refined Scatter Plot (V2)**

Let's fix our scatter plot using `price_log`, adding opacity for overplotting and coloring by `grade` to provide more context for Sarah.

#### Effect of Living Area on Price

In [ ]:
scatter_plot_v2 = alt.Chart(housing_cleaned_df).mark_circle(opacity=0.6, filled=False).encode(
    x=alt.X('sqft_living', title='Living Area Sq.Ft', scale=alt.Scale(zero=False)),
    y=alt.Y('price', title='Sale Price $', scale=alt.Scale(zero=False)),
    # facet=alt.Facet('bed_cat:N', title='Number of Bedrooms'),
    color=alt.Color('bed_cat:N', title='Bedrooms'),
    tooltip=[
        alt.Tooltip('price:Q', title='Price', format='$,.0f'),
        alt.Tooltip('sqft_living:Q', title='SqFt Living'),
        alt.Tooltip('bedrooms:Q', title='Bedrooms'),
        alt.Tooltip('bathrooms:Q', title='Bathrooms'),
        alt.Tooltip('grade:O', title='Grade')
    ]
).transform_filter(
   alt.FieldOneOfPredicate(field='bed_cat', oneOf="1+ 2+ 3+ 4+".split())
).properties(
    title='Price vs. Living Area, Colored by bedrooms',
    height=400,
    width=600
).resolve_scale(
    x='independent'
)
scatter_plot_log = alt.Chart(housing_cleaned_df).mark_circle(opacity=0.6, filled=False).encode(
    x=alt.X('sqft_living_log', title='Living Area Sq.Ft (Log Scale)', scale=alt.Scale(zero=False)),
    y=alt.Y('price_log', title='Sale Price $ (Log Scale)', scale=alt.Scale(zero=False)),
    # facet=alt.Facet('bed_cat:N', title='Number of Bedrooms'),
    color=alt.Color('bed_cat:N', title='Bedrooms'),
    tooltip=[
        alt.Tooltip('price:Q', title='Price', format='$,.0f'),
        alt.Tooltip('sqft_living:Q', title='SqFt Living'),
        alt.Tooltip('bedrooms:Q', title='Bedrooms'),
        alt.Tooltip('bathrooms:Q', title='Bathrooms'),
        alt.Tooltip('grade:O', title='Grade')
    ]
).transform_filter(
   alt.FieldOneOfPredicate(field='bed_cat', oneOf="1+ 2+ 3+ 4+".split())
).properties(
    title='Price vs. Living Area, Colored by bedrooms',
    height=400,
    width=600
).resolve_scale(
    x='independent'
)

display(scatter_plot_v2 & scatter_plot_log)

#### Effect of Beds & Baths on Price per Sq.Ft

In [ ]:
alt.Chart(housing_cleaned_df).mark_bar().encode(
    x=alt.X('bed_cat', title=None),
    y=alt.Y('median(price_per_sqft)', title='Median Price per Sqft.'),
    color=alt.Color('bed_cat:N', title='Number of Bedrooms'),
    tooltip=[
        alt.Tooltip('price:Q', title='Price', format='$,.0f'),
        alt.Tooltip('sqft_living:Q', title='SqFt Living'),
        alt.Tooltip('bedrooms:Q', title='Bedrooms'),
        alt.Tooltip('bathrooms:Q', title='Bathrooms'),
        alt.Tooltip('grade:O', title='Grade')
    ]
).properties(
    title='Effect of Beds and Baths on $ per Sq.ft.',
).transform_filter(
   alt.FieldOneOfPredicate(field='bed_cat', oneOf="1+ 2+ 3+ 4+".split())
).facet(alt.Facet('bath_cat', title='Number of Bathrooms')
)

#### Effect of Grade on Price per Sq.Ft

In [ ]:
grade_string = alt.Chart(housing_cleaned_df).mark_bar().encode(
    x=alt.X('grade'),
    y=alt.Y('median(price_per_sqft)', title='Median Price per Sqft.')
).properties(
    title='Effect of Grade on $ per Sq.ft.',
)

grade_enum = alt.Chart(housing_cleaned_df).mark_bar().encode(
    x=alt.X('grade_cat'),
    y=alt.Y('median(price_per_sqft)', title='Median Price per Sqft.')
).properties(
    title='Effect of Grade on $ per Sq.ft.',
)

grade_string | grade_enum

**New Chart: Price Distribution Histogram (V2)**

Now we can create the histogram of prices to answer Sarah's first question. We'll use the original `price` column but use a log scale on the x-axis for better readability.

In [ ]:
price_histogram = alt.Chart(housing_cleaned_df).mark_bar().encode(
    x=alt.X('price_log:Q').bin(maxbins=30).title('Sale Price (log USD)'),
    y=alt.Y('count()').title('Number of Houses'),
    tooltip=[
        alt.Tooltip('price', title="Price USD"),
        alt.Tooltip('count()', title='Number of Houses'), ]
).properties(
    title='Distribution of House Prices (Log Scale)',
    width=500,
    height=350
)

display(price_histogram)

---
## Phase 4: Building the Narrative - Horizontal Development

We have refined individual charts ("vertical development"). Now let's arrange them into a story ("horizontal development").

### Step 8: Creating a Cohesive Set of Visuals (The "Comic Strip")

We'll create a story for Sarah:
1.  Start with the overall price distribution (Histogram).
2.  Then, show what drives price (Scatter plot of price vs. size/grade).

We can use Altair's `&` (vconcat) or `|` (hconcat) operator to combine them. Let's stack them vertically (`vconcat`).

In [ ]:
# Combine the two charts into a single story view
narrative_v1 = price_histogram & scatter_plot_v2

display(narrative_v1)

### Step 9: Adding Narrative Text and Annotations

A visual layout is a good start, but it's not a full story. We need text and annotations to guide Sarah. While we can't add extensive prose between charts in a concatenated view, we can improve titles and add annotations.

Let's add a median price line to our histogram to give Sarah a clear anchor point for what's "typical".

In [ ]:
# Calculate median price for annotation
median_price = housing_cleaned_df['price_log'].median()

# Create the median line using mark_rule
median_line = alt.Chart(pd.DataFrame({'median_price': [median_price]})) \
    .mark_rule(color='red', strokeDash=[3,3], size=2) \
    .encode(x='median_price:Q')

# Refine titles and layer the median line onto the histogram
price_histogram_final = alt.Chart(housing_cleaned_df).mark_bar().encode(
    x=alt.X('price:Q', bin=alt.Bin(maxbins=20), title='Sale Price (USD)', scale=alt.Scale(type="log")),
    y=alt.Y('count()', title='Number of Houses'),
    tooltip=[alt.Tooltip('count()', title='Number of Houses')]
).properties(
    title={
        "text": "1. Most House Prices Cluster Around the Median",
        "subtitle": "King County has many mid-range homes and a long tail of high-value properties."
    },
    width=500,
    height=300
)
price_histogram_final

In [ ]:

# Refine scatter plot titles
scatter_plot_final = alt.Chart(housing_cleaned_df).mark_point(opacity=0.4, filled=True).encode(
    x=alt.X('sqft_living', title='Living Area (sq ft)', scale=alt.Scale(zero=False)),
    y=alt.Y('price_log', title='Sale Price (Log Scale)'),
    color=alt.Color('grade:O', title='Construction Grade', scale=alt.Scale(scheme='viridis')),
    tooltip=[
        alt.Tooltip('price:Q', title='Price', format='$,.0f'),
        alt.Tooltip('sqft_living:Q', title='SqFt Living'),
        alt.Tooltip('grade:O', title='Grade')
    ]
).properties(
    title={
        "text": "2. Price is Driven by Size and Quality",
        "subtitle": "Larger homes and those with a higher construction grade command higher prices."
    },
    width=500,
    height=300
).interactive()


# The final story combines the annotated histogram with the refined scatter plot
final_story = (price_histogram_final + median_line) & scatter_plot_final

In [ ]:
# Calculate median price for annotation
median_price = housing_cleaned_df['price_log'].median()

# Create the median line using mark_rule
median_line = alt.Chart(pd.DataFrame({'median_price': [median_price]})) \
    .mark_rule(color='red', strokeDash=[3,3], size=2) \
    .encode(x='median_price:Q')

# Refine titles and layer the median line onto the histogram
price_histogram_final = alt.Chart(housing_cleaned_df).mark_bar().encode(
    x=alt.X('price_log:Q', bin=alt.Bin(maxbins=20), title='Sale Price (USD)'),
    y=alt.Y('count()', title='Number of Houses'),
    tooltip=[alt.Tooltip('count()', title='Number of Houses')]
).properties(
    title={
        "text": "1. Most House Prices Cluster Around the Median",
        "subtitle": "King County has many mid-range homes and a long tail of high-value properties."
    },
    width=500,
    height=300
)

# Refine scatter plot titles
scatter_plot_final = alt.Chart(housing_cleaned_df).mark_point(opacity=0.4, filled=True).encode(
    x=alt.X('sqft_living', title='Living Area (sq ft)', scale=alt.Scale(zero=False)),
    y=alt.Y('price_log', title='Sale Price (Log Scale)'),
    color=alt.Color('grade:O', title='Construction Grade', scale=alt.Scale(scheme='viridis')),
    tooltip=[
        alt.Tooltip('price:Q', title='Price', format='$,.0f'),
        alt.Tooltip('sqft_living:Q', title='SqFt Living'),
        alt.Tooltip('grade:O', title='Grade')
    ]
).properties(
    title={
        "text": "2. Price is Driven by Size and Quality",
        "subtitle": "Larger homes and those with a higher construction grade command higher prices."
    },
    width=500,
    height=300
).interactive()


# The final story combines the annotated histogram with the refined scatter plot
final_story = (price_histogram_final + median_line) & scatter_plot_final

In [ ]:
final_story